In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from scipy import stats
from creditcard_psp.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

from creditcard_psp.dataset import load_transactions, merge_service_fees, assign_transaction_ids, split_train_test
from creditcard_psp.features import add_time_features, encode_and_scale

input_path  = RAW_DATA_DIR / "PSP_Jan_Feb_2019.xlsx"
fee_path    = RAW_DATA_DIR / "PSP_Servicegebuehren.xlsx"
output_path = PROCESSED_DATA_DIR / "PSP_Jan_Feb_2019_processed.csv"

2025-04-21 21:09:28.406 | INFO     | creditcard_psp.config:<module>:11 - PROJ_ROOT path is: C:\Users\miria\creditcard_psp


In [2]:
# XLSX-Datei einlesen
df = load_transactions(input_path)

# Anzahl Zeilen
print(df.count())

# Anzahl Duplikate
duplicates = df.duplicated().sum()
print("Anzahl Duplikate: ", duplicates)

# Duplikate löschen
df = df.drop_duplicates()

# Anzahl Zeilen
print(df.count())

2025-04-21 21:09:38.235 | INFO     | creditcard_psp.dataset:load_transactions:28 - Loading transactions from C:\Users\miria\creditcard_psp\data\raw\PSP_Jan_Feb_2019.xlsx
Unnamed: 0    50410
tmsp          50410
country       50410
amount        50410
success       50410
PSP           50410
3D_secured    50410
card          50410
dtype: int64
Anzahl Duplikate:  0
Unnamed: 0    50410
tmsp          50410
country       50410
amount        50410
success       50410
PSP           50410
3D_secured    50410
card          50410
dtype: int64


In [3]:
# Merge Service-Fees
df = merge_service_fees(df, fee_path)

2025-04-21 21:10:30.012 | INFO     | creditcard_psp.dataset:merge_service_fees:112 - Merging service fees from C:\Users\miria\creditcard_psp\data\raw\PSP_Servicegebuehren.xlsx
2025-04-21 21:10:30.113 | WARNING  | creditcard_psp.dataset:merge_service_fees:128 - Missing fee entries after merge: {'fee_successful': 0, 'fee_not_successful': 0}


In [4]:
# Extract weekday, hour, minute
df = add_time_features(df)

In [5]:
# Adds transaction_id, transaction_success, attempt_number
df = assign_transaction_ids(df)      

In [6]:
# Split into train and testset 
X_train, X_test, y_train, y_test = split_train_test(
    df, target_col='transaction_success', test_size=0.2, stratify=True
)

In [7]:
# Encoding and scaling
df = encode_and_scale(
    df,
    categorical_cols=['PSP', 'card', 'country'],
    amount_col='amount',
    drop_first=True
)

df.head(10)

,Unnamed: 0,tmsp,amount,success,3D_secured,fee_successful,fee_not_successful,weekday,hour,minute,...,attempt_number,PSP_Moneycard,PSP_Simplecard,PSP_UK_Card,card_Master,card_Visa,country_Germany,country_Switzerland,amount_log,amount_scaled
9238,9238,2019-01-10 03:49:12,6,0,0,5.0,2.0,3,3,49,...,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.945910,-4.931956
9239,9239,2019-01-10 03:49:37,6,0,0,1.0,0.5,3,3,49,...,2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.945910,-4.931956
22742,22742,2019-01-27 14:01:11,6,1,0,1.0,0.5,6,14,1,...,1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.945910,-4.931956
33737,33737,2019-02-08 05:02:33,6,0,0,3.0,1.0,4,5,2,...,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.945910,-4.931956
33738,33738,2019-02-08 05:02:37,6,0,0,3.0,1.0,4,5,2,...,2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.945910,-4.931956
33739,33739,2019-02-08 05:02:39,6,0,0,1.0,0.5,4,5,2,...,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.945910,-4.931956
40539,40539,2019-02-16 08:24:40,6,1,1,3.0,1.0,5,8,24,...,1,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.945910,-4.931956
21179,21179,2019-01-25 04:18:26,7,0,0,5.0,2.0,4,4,18,...,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.079442,-4.726737
23798,23798,2019-01-28 23:48:19,7,0,0,3.0,1.0,0,23,48,...,1,0.0,0.0,1.0,1.0,0.0,0.0,0.0,2.079442,-4.726737
31712,31712,2019-02-06 04:09:05,7,0,0,3.0,1.0,2,4,9,...,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.079442,-4.726737


In [8]:
# Z‑Score berechnen 
log_vals = df['amount_log']
z_scores = np.abs(stats.zscore(log_vals))

# Ausreißer zählen (Z‑Score > 3)
n_outliers_log = (z_scores > 3).sum()
print(f"Ausreißer nach Log-Transformation (Z>3): {n_outliers_log}")

Ausreißer nach Log-Transformation (Z>3): 792


In [9]:
Q1 = df['amount_log'].quantile(0.25)
Q3 = df['amount_log'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = df[(df['amount_log'] < lower) | (df['amount_log'] > upper)]
print(f"Ausreißer nach IQR (log space): {len(outliers_iqr)}")

Ausreißer nach IQR (log space): 2885


In [10]:
# df speichern
df.to_pickle('df.pkl')